# Fundamentals of Optimization and Regularization in Deep Learning

This notebook provides a comprehensive visual guide to optimization and regularization techniques used in deep learning. We will explore:

- **Convex vs Non-convex Optimization**: Understanding the landscape of loss functions
- **Gradient Descent Variants**: Batch GD, SGD, Mini-batch GD
- **Learning Rate Effects**: How learning rate impacts convergence
- **Optimization Algorithms**: SGD with Momentum, Adam, RMSprop
- **Learning Rate Schedulers**: Learning Rate decay comparisons
- **Regularization**: L1, L2, Dropout, Data Augmentation

> **Note**: This notebook uses **PyTorch** as the primary deep learning framework.


# Part 0: Library Imports and Plotting Functions


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import warnings
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from PIL import Image
import os
import subprocess
import sys

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- 1. Graphviz & Torchviz Portable Setup ---
# Installs python dependencies and registers dot.exe in environment PATH dynamically
try:
    import torchviz
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchviz", "graphviz"])
    import torchviz

def setup_graphviz():
    for path in os.environ["PATH"].split(os.pathsep):
        if os.path.exists(os.path.join(path, "dot.exe")):
            return True

    common_dirs = [
        r"C:\Program Files\Graphviz\bin",
        r"C:\Program Files (x86)\Graphviz\bin",
        r"C:\Program Files\Graphviz",
        r"C:\Program Files (x86)\Graphviz",
    ]
    for d in common_dirs:
        if os.path.exists(os.path.join(d, "dot.exe")):
            os.environ["PATH"] += os.pathsep + d
            return True

    try:
        subprocess.check_call(["winget", "install", "--id", "Graphviz.Graphviz", "--silent", "--accept-source-agreements", "--accept-package-agreements"])
        import glob
        matches = glob.glob(r"C:\Program Files\Graphviz*\bin")
        if matches:
            os.environ["PATH"] += os.pathsep + matches[0]
            return True
    except Exception:
        pass
    return False

setup_graphviz()

# --- 2. Math Expression Parsers & Autograd Calculators ---
def parse_expression_1d(expr_str):
    safe_dict = {
        'x': None,
        'torch': torch,
        'np': np,
        'sin': torch.sin,
        'cos': torch.cos,
        'tan': torch.tan,
        'exp': torch.exp,
        'log': torch.log,
        'sinh': torch.sinh,
        'cosh': torch.cosh,
        'pi': np.pi
    }
    def f(x):
        local_dict = safe_dict.copy()
        local_dict['x'] = x
        return eval(expr_str, {"__builtins__": None}, local_dict)
    return f

def parse_expression_2d(expr_str):
    safe_dict = {
        'x': None,
        'y': None,
        'torch': torch,
        'np': np,
        'sin': torch.sin,
        'cos': torch.cos,
        'tan': torch.tan,
        'exp': torch.exp,
        'log': torch.log,
        'pi': np.pi
    }
    def f(x, y):
        local_dict = safe_dict.copy()
        local_dict['x'] = x
        local_dict['y'] = y
        return eval(expr_str, {"__builtins__": None}, local_dict)
    return f

# --- 3. Numerical Minima Detection Helpers ---
def find_minima_1d(f, x_range=(-8, 8)):
    x_grid = np.linspace(x_range[0], x_range[1], 1000)
    x_tensor = torch.tensor(x_grid, dtype=torch.float32)
    try:
        with torch.no_grad():
            y_grid = f(x_tensor).numpy()
    except Exception:
        y_grid = f(x_grid)

    glob_idx = np.argmin(y_grid)
    glob_min = (x_grid[glob_idx], y_grid[glob_idx])

    local_mins = []
    for i in range(1, len(y_grid) - 1):
        if y_grid[i] < y_grid[i-1] and y_grid[i] < y_grid[i+1]:
            if abs(x_grid[i] - glob_min[0]) > 0.5:
                local_mins.append((x_grid[i], y_grid[i]))
    return glob_min, local_mins

def find_minima_2d(Z, X, Y):
    glob_idx = np.argmin(Z)
    r_g, c_g = np.unravel_index(glob_idx, Z.shape)
    glob_min = (X[r_g, c_g], Y[r_g, c_g], Z[r_g, c_g])

    local_mins = []
    for r in range(1, Z.shape[0] - 1):
        for c in range(1, Z.shape[1] - 1):
            val = Z[r, c]
            if (val < Z[r-1, c] and val < Z[r+1, c] and
                val < Z[r, c-1] and val < Z[r, c+1] and
                val < Z[r-1, c-1] and val < Z[r-1, c+1] and
                val < Z[r+1, c-1] and val < Z[r+1, c+1]):
                x_val, y_val = X[r, c], Y[r, c]
                if np.sqrt((x_val - glob_min[0])**2 + (y_val - glob_min[1])**2) > 0.8:
                    local_mins.append((x_val, y_val, val))
    return glob_min, local_mins

# --- 4. Math Expression Parsers & Autograd Calculators ---
def optimize_function_1d_torch(f_1d, start_x, lr, epochs, optimizer_name='SGD'):
    x = torch.tensor(float(start_x), requires_grad=True)
    if optimizer_name == 'SGD':
        optimizer = optim.SGD([x], lr=lr)
    elif optimizer_name == 'Momentum':
        optimizer = optim.SGD([x], lr=lr, momentum=0.9)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop([x], lr=lr)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam([x], lr=lr)

    path_x = [x.item()]
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = f_1d(x)
        loss.backward()
        optimizer.step()
        path_x.append(x.item())
    return np.array(path_x)

def optimize_function_2d_torch(f_2d, start_x, start_y, lr, epochs, optimizer_name='SGD'):
    params = torch.tensor([float(start_x), float(start_y)], requires_grad=True)
    if optimizer_name == 'SGD':
        optimizer = optim.SGD([params], lr=lr)
    elif optimizer_name == 'Momentum':
        optimizer = optim.SGD([params], lr=lr, momentum=0.9)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop([params], lr=lr)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam([params], lr=lr)

    path_x, path_y = [params[0].item()], [params[1].item()]
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = f_2d(params[0], params[1])
        loss.backward()
        optimizer.step()
        path_x.append(params[0].item())
        path_y.append(params[1].item())
    return np.array(path_x), np.array(path_y)



# --- 5. Plotting & Visualizer Helpers ---
def setup_plot(title, xlabel, ylabel, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.6)
    return ax

def plot_1d_trajectory(ax, x_vals, f_func, path_x, title, color='red', label='Trajectory'):
    ax.clear()
    setup_plot(title, 'x', 'f(x)', ax)
    y_vals = f_func(x_vals)
    ax.plot(x_vals, y_vals, color='black', alpha=0.3, label='Loss Landscape')

    # Highlight global and local minima
    glob_min, local_mins = find_minima_1d(f_func, (x_vals[0], x_vals[-1]))
    ax.scatter(glob_min[0], glob_min[1], color='red', marker='*', s=150, zorder=6, label=f'Global Min ({glob_min[0]:.2f})')
    for idx, lm in enumerate(local_mins):
        ax.scatter(lm[0], lm[1], color='orange', marker='*', s=120, zorder=6, label=f'Local Min {idx+1} ({lm[0]:.2f})' if idx==0 else '')

    path_y = f_func(path_x)
    ax.plot(path_x, path_y, color=color, alpha=0.6, linestyle='-', marker='o', label=label)
    ax.scatter(path_x[0], path_y[0], color='green', s=100, label='Start Point', zorder=5)
    ax.scatter(path_x[-1], path_y[-1], color='red', s=100, label='Final Point', zorder=5)
    ax.legend()

def plot_2d_trajectory(ax, X, Y, Z, path_x, path_y, title, color='blue', label='Optimizer Path'):
    ax.clear()
    ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.6)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    # Highlight global and local minima
    glob_min, local_mins = find_minima_2d(Z, X, Y)
    ax.scatter(glob_min[0], glob_min[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
    for lm in local_mins:
        ax.scatter(lm[0], lm[1], color='orange', marker='*', s=140, edgecolor='black', zorder=6, label='Local Min')

    ax.plot(path_x, path_y, color=color, alpha=0.8, marker='.', markersize=6, label=label)
    ax.scatter(path_x[0], path_y[0], color='green', s=100, label='Start', zorder=5)
    ax.scatter(path_x[-1], path_y[-1], color='red', s=120, marker='x', label='End', zorder=5)
    ax.legend()

def plot_1d_contour_surface_viewer(expr_1d, expr_2d):
    try:
        f1d = parse_expression_1d(expr_1d)
        f2d = parse_expression_2d(expr_2d)
        fig = plt.figure(figsize=(18, 5))

        # 1. 1D
        ax1 = fig.add_subplot(1, 3, 1)
        x_1d = np.linspace(-8, 8, 400)
        x_1d_tensor = torch.tensor(x_1d, dtype=torch.float32)
        with torch.no_grad():
            y_1d = f1d(x_1d_tensor).numpy()
        setup_plot(f'1D Function: {expr_1d}', 'x', 'f(x)', ax1)
        ax1.plot(x_1d, y_1d, color='blue', linewidth=2)

        # Highlight global and local minima
        glob_min1d, local_mins1d = find_minima_1d(f1d, (-8, 8))
        ax1.scatter(glob_min1d[0], glob_min1d[1], color='red', marker='*', s=150, zorder=6, label='Global Min')
        for lm in local_mins1d:
            ax1.scatter(lm[0], lm[1], color='orange', marker='*', s=120, zorder=6, label='Local Min')
        ax1.legend()

        # 2. 2D Surface
        ax2 = fig.add_subplot(1, 3, 2, projection='3d')
        x_2d = np.linspace(-5, 5, 50)
        y_2d = np.linspace(-5, 5, 50)
        X, Y = np.meshgrid(x_2d, y_2d)
        X_tensor = torch.tensor(X, dtype=torch.float32)
        Y_tensor = torch.tensor(Y, dtype=torch.float32)
        with torch.no_grad():
            Z = f2d(X_tensor, Y_tensor).numpy()
        ax2.plot_surface(X, Y, Z, cmap='viridis', alpha=0.7)
        ax2.set_title(f'2D Surface: {expr_2d}', fontsize=10, fontweight='bold')
        ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('f(x,y)')

        # 3. 2D Contour
        ax3 = fig.add_subplot(1, 3, 3)
        ax3.contour(X, Y, Z, levels=35, cmap='magma')
        ax3.set_title(f'2D Contour: {expr_2d}', fontsize=10, fontweight='bold')
        ax3.set_xlabel('x'); ax3.set_ylabel('y')

        # Highlight global and local minima
        glob_min2d, local_mins2d = find_minima_2d(Z, X, Y)
        ax3.scatter(glob_min2d[0], glob_min2d[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
        for lm in local_mins2d:
            ax3.scatter(lm[0], lm[1], color='orange', marker='*', s=140, edgecolor='black', zorder=6, label='Local Min')
        ax3.legend()
        ax3.grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error rendering function: {e}")

def plot_interactive_optimizer(expr_1d, start_x, lr_1d, opt_1d, expr_2d, start_y_x, start_y_y, lr_2d, opt_2d, epochs):
    try:
        f1d = parse_expression_1d(expr_1d)
        f2d = parse_expression_2d(expr_2d)
        path_1d = optimize_function_1d_torch(f1d, start_x, lr_1d, epochs, opt_1d)
        path_2d_x, path_2d_y = optimize_function_2d_torch(f2d, start_y_x, start_y_y, lr_2d, epochs, opt_2d)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        x_vals = np.linspace(-8, 8, 400)
        plot_1d_trajectory(axes[0], x_vals, lambda x: f1d(torch.tensor(x)).numpy(), path_1d,
                           f"1D Optimization ({opt_1d} on '{expr_1d}')", color='orange', label=f"Path: {opt_1d}")

        x_2d = np.linspace(-5, 5, 100)
        y_2d = np.linspace(-5, 5, 100)
        X, Y = np.meshgrid(x_2d, y_2d)
        X_tensor = torch.tensor(X, dtype=torch.float32)
        Y_tensor = torch.tensor(Y, dtype=torch.float32)
        with torch.no_grad():
            Z = f2d(X_tensor, Y_tensor).numpy()
        plot_2d_trajectory(axes[1], X, Y, Z, path_2d_x, path_2d_y,
                           f"2D Optimization ({opt_2d} on '{expr_2d}')", color='red', label=f"Path: {opt_2d}")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error in optimizer playground: {e}")

def get_2x2_lr_animation():
    f_1d_c = lambda x: x**2
    f_1d_nc = lambda x: x**2 + 10 * torch.sin(x)
    f_2d_c = lambda x, y: x**2 + y**2
    f_2d_nc = lambda x, y: x**2 + y**2 + 10 * torch.sin(x) * torch.cos(y)
    lrs = [0.01, 0.05, 0.25]
    colors = ['green', 'orange', 'red']
    epochs_anim = 30
    paths_1d_c = [optimize_function_1d_torch(f_1d_c, -7.0, lr, epochs_anim, 'SGD') for lr in lrs]
    paths_1d_nc = [optimize_function_1d_torch(f_1d_nc, -7.0, lr, epochs_anim, 'SGD') for lr in lrs]
    paths_2d_c = [optimize_function_2d_torch(f_2d_c, -4.0, -4.0, lr, epochs_anim, 'SGD') for lr in lrs]
    paths_2d_nc = [optimize_function_2d_torch(f_2d_nc, 4.0, 4.0, lr, epochs_anim, 'SGD') for lr in lrs]
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    x_1d = np.linspace(-8, 8, 400)
    x_2d = np.linspace(-5, 5, 100)
    y_2d = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x_2d, y_2d)
    Z_c = X**2 + Y**2
    Z_nc = X**2 + Y**2 + 10 * np.sin(X) * np.cos(Y)
    lines_1d_c = [axes[0, 0].plot([], [], marker='o', color=colors[i], alpha=0.7, label=f'LR = {lrs[i]}')[0] for i in range(3)]
    lines_1d_nc = [axes[0, 1].plot([], [], marker='o', color=colors[i], alpha=0.7, label=f'LR = {lrs[i]}')[0] for i in range(3)]
    lines_2d_c = [axes[1, 0].plot([], [], marker='o', color=colors[i], alpha=0.8, label=f'LR = {lrs[i]}')[0] for i in range(3)]
    lines_2d_nc = [axes[1, 1].plot([], [], marker='o', color=colors[i], alpha=0.8, label=f'LR = {lrs[i]}')[0] for i in range(3)]

    # Plot background functions & Highlight Minima
    axes[0, 0].plot(x_1d, x_1d**2, color='black', alpha=0.2)
    glob_1dc, _ = find_minima_1d(f_1d_c, (-8, 8))
    axes[0, 0].scatter(glob_1dc[0], glob_1dc[1], color='red', marker='*', s=150, zorder=6, label='Global Min')
    setup_plot('1D Convex (x^2)', 'x', 'f(x)', axes[0, 0]); axes[0, 0].legend()

    axes[0, 1].plot(x_1d, x_1d**2 + 10 * np.sin(x_1d), color='black', alpha=0.2)
    glob_1dnc, locals_1dnc = find_minima_1d(f_1d_nc, (-8, 8))
    axes[0, 1].scatter(glob_1dnc[0], glob_1dnc[1], color='red', marker='*', s=150, zorder=6, label='Global Min')
    for lm in locals_1dnc:
        axes[0, 1].scatter(lm[0], lm[1], color='orange', marker='*', s=120, zorder=6, label='Local Min')
    setup_plot('1D Non-Convex (x^2 + 10*sin(x))', 'x', 'f(x)', axes[0, 1]); axes[0, 1].legend()

    axes[1, 0].contour(X, Y, Z_c, levels=25, cmap='viridis', alpha=0.4)
    glob_2dc, _ = find_minima_2d(Z_c, X, Y)
    axes[1, 0].scatter(glob_2dc[0], glob_2dc[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
    setup_plot('2D Convex (Contour)', 'x', 'y', axes[1, 0]); axes[1, 0].legend()

    axes[1, 1].contour(X, Y, Z_nc, levels=25, cmap='magma', alpha=0.4)
    glob_2dnc, locals_2dnc = find_minima_2d(Z_nc, X, Y)
    axes[1, 1].scatter(glob_2dnc[0], glob_2dnc[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
    for lm in locals_2dnc:
        axes[1, 1].scatter(lm[0], lm[1], color='orange', marker='*', s=140, edgecolor='black', zorder=6, label='Local Min')
    setup_plot('2D Non-Convex (Contour)', 'x', 'y', axes[1, 1]); axes[1, 1].legend()

    def animate(frame):
        for i in range(3):
            xc = paths_1d_c[i][:frame+1]
            lines_1d_c[i].set_data(xc, xc**2)
            xnc = paths_1d_nc[i][:frame+1]
            lines_1d_nc[i].set_data(xnc, xnc**2 + 10 * np.sin(xnc))
            px_c, py_c = paths_2d_c[i][0][:frame+1], paths_2d_c[i][1][:frame+1]
            lines_2d_c[i].set_data(px_c, py_c)
            px_nc, py_nc = paths_2d_nc[i][0][:frame+1], paths_2d_nc[i][1][:frame+1]
            lines_2d_nc[i].set_data(px_nc, py_nc)
        return lines_1d_c + lines_1d_nc + lines_2d_c + lines_2d_nc
    anim = FuncAnimation(fig, animate, frames=epochs_anim+1, interval=150, blit=True)
    plt.close()
    return anim

def get_2x2_sp_animation():
    f_1d_c = lambda x: x**2
    f_1d_nc = lambda x: x**2 + 10 * torch.sin(x)
    f_2d_c = lambda x, y: x**2 + y**2
    f_2d_nc = lambda x, y: x**2 + y**2 + 10 * torch.sin(x) * torch.cos(y)
    lr_fixed = 0.05
    sp_1d = [-7.0, 1.0, 6.0]
    sp_2d = [(-4.0, -4.0), (1.0, 1.0), (4.0, 4.0)]
    colors_sp = ['blue', 'purple', 'brown']
    epochs_anim = 30
    paths_sp_1d_c = [optimize_function_1d_torch(f_1d_c, sp, lr_fixed, epochs_anim, 'SGD') for sp in sp_1d]
    paths_sp_1d_nc = [optimize_function_1d_torch(f_1d_nc, sp, lr_fixed, epochs_anim, 'SGD') for sp in sp_1d]
    paths_sp_2d_c = [optimize_function_2d_torch(f_2d_c, sp[0], sp[1], lr_fixed, epochs_anim, 'SGD') for sp in sp_2d]
    paths_sp_2d_nc = [optimize_function_2d_torch(f_2d_nc, sp[0], sp[1], lr_fixed, epochs_anim, 'SGD') for sp in sp_2d]
    fig_sp, axes_sp = plt.subplots(2, 2, figsize=(14, 12))
    x_1d = np.linspace(-8, 8, 400)
    x_2d = np.linspace(-5, 5, 100)
    y_2d = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x_2d, y_2d)
    Z_c = X**2 + Y**2
    Z_nc = X**2 + Y**2 + 10 * np.sin(X) * np.cos(Y)
    lines_sp_1d_c = [axes_sp[0, 0].plot([], [], marker='o', color=colors_sp[i], alpha=0.7, label=f'Start = {sp_1d[i]}')[0] for i in range(3)]
    lines_sp_1d_nc = [axes_sp[0, 1].plot([], [], marker='o', color=colors_sp[i], alpha=0.7, label=f'Start = {sp_1d[i]}')[0] for i in range(3)]
    lines_sp_2d_c = [axes_sp[1, 0].plot([], [], marker='o', color=colors_sp[i], alpha=0.8, label=f'Start = {sp_2d[i]}')[0] for i in range(3)]
    lines_sp_2d_nc = [axes_sp[1, 1].plot([], [], marker='o', color=colors_sp[i], alpha=0.8, label=f'Start = {sp_2d[i]}')[0] for i in range(3)]

    # Highlight Minima
    axes_sp[0, 0].plot(x_1d, x_1d**2, color='black', alpha=0.2)
    glob_1dc, _ = find_minima_1d(f_1d_c, (-8, 8))
    axes_sp[0, 0].scatter(glob_1dc[0], glob_1dc[1], color='red', marker='*', s=150, zorder=6, label='Global Min')
    setup_plot('1D Convex (x^2)', 'x', 'f(x)', axes_sp[0, 0]); axes_sp[0, 0].legend()

    axes_sp[0, 1].plot(x_1d, x_1d**2 + 10 * np.sin(x_1d), color='black', alpha=0.2)
    glob_1dnc, locals_1dnc = find_minima_1d(f_1d_nc, (-8, 8))
    axes_sp[0, 1].scatter(glob_1dnc[0], glob_1dnc[1], color='red', marker='*', s=150, zorder=6, label='Global Min')
    for lm in locals_1dnc:
        axes_sp[0, 1].scatter(lm[0], lm[1], color='orange', marker='*', s=120, zorder=6, label='Local Min')
    setup_plot('1D Non-Convex (x^2 + 10*sin(x))', 'x', 'f(x)', axes_sp[0, 1]); axes_sp[0, 1].legend()

    axes_sp[1, 0].contour(X, Y, Z_c, levels=25, cmap='viridis', alpha=0.4)
    glob_2dc, _ = find_minima_2d(Z_c, X, Y)
    axes_sp[1, 0].scatter(glob_2dc[0], glob_2dc[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
    setup_plot('2D Convex (Contour)', 'x', 'y', axes_sp[1, 0]); axes_sp[1, 0].legend()

    axes_sp[1, 1].contour(X, Y, Z_nc, levels=25, cmap='magma', alpha=0.4)
    glob_2dnc, locals_2dnc = find_minima_2d(Z_nc, X, Y)
    axes_sp[1, 1].scatter(glob_2dnc[0], glob_2dnc[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
    for lm in locals_2dnc:
        axes_sp[1, 1].scatter(lm[0], lm[1], color='orange', marker='*', s=140, edgecolor='black', zorder=6, label='Local Min')
    setup_plot('2D Non-Convex (Contour)', 'x', 'y', axes_sp[1, 1]); axes_sp[1, 1].legend()

    def animate_sp(frame):
        for i in range(3):
            xc = paths_sp_1d_c[i][:frame+1]
            lines_sp_1d_c[i].set_data(xc, xc**2)
            xnc = paths_sp_1d_nc[i][:frame+1]
            lines_sp_1d_nc[i].set_data(xnc, xnc**2 + 10 * np.sin(xnc))
            px_c, py_c = paths_sp_2d_c[i][0][:frame+1], paths_sp_2d_c[i][1][:frame+1]
            lines_sp_2d_c[i].set_data(px_c, py_c)
            px_nc, py_nc = paths_sp_2d_nc[i][0][:frame+1], paths_sp_2d_nc[i][1][:frame+1]
            lines_sp_2d_nc[i].set_data(px_nc, py_nc)
        return lines_sp_1d_c + lines_sp_1d_nc + lines_sp_2d_c + lines_sp_2d_nc
    anim_sp = FuncAnimation(fig_sp, animate_sp, frames=epochs_anim+1, interval=150, blit=True)
    plt.close()
    return anim_sp

def get_optimizers_comparison_animation():
    f_2d_nc = lambda x, y: x**2 + y**2 + 10 * torch.sin(x) * torch.cos(y)
    start_opt = (4.0, 4.0)
    epochs_opt = 40
    opt_lrs = {'SGD': 0.05, 'Momentum': 0.02, 'RMSprop': 0.05, 'Adam': 0.15}
    colors_opt = {'SGD': 'blue', 'Momentum': 'red', 'RMSprop': 'green', 'Adam': 'purple'}
    paths_opt = {}
    paths_opt['SGD'] = optimize_function_2d_torch(f_2d_nc, start_opt[0], start_opt[1], opt_lrs['SGD'], epochs_opt, 'SGD')
    paths_opt['Momentum'] = optimize_function_2d_torch(f_2d_nc, start_opt[0], start_opt[1], opt_lrs['Momentum'], epochs_opt, 'Momentum')
    paths_opt['RMSprop'] = optimize_function_2d_torch(f_2d_nc, start_opt[0], start_opt[1], opt_lrs['RMSprop'], epochs_opt, 'RMSprop')
    paths_opt['Adam'] = optimize_function_2d_torch(f_2d_nc, start_opt[0], start_opt[1], opt_lrs['Adam'], epochs_opt, 'Adam')
    fig_opt, ax_opt = plt.subplots(figsize=(8, 8))
    x_2d = np.linspace(-5, 5, 100)
    y_2d = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x_2d, y_2d)
    Z_nc = X**2 + Y**2 + 10 * np.sin(X) * np.cos(Y)
    ax_opt.contour(X, Y, Z_nc, levels=30, cmap='magma', alpha=0.5)
    ax_opt.scatter(*start_opt, color='black', s=120, label='Start Point', zorder=5)

    # Highlight Minima
    glob_2dnc, locals_2dnc = find_minima_2d(Z_nc, X, Y)
    ax_opt.scatter(glob_2dnc[0], glob_2dnc[1], color='red', marker='*', s=180, edgecolor='black', zorder=6, label='Global Min')
    for lm in locals_2dnc:
        ax_opt.scatter(lm[0], lm[1], color='orange', marker='*', s=140, edgecolor='black', zorder=6, label='Local Min')

    lines_opt = [ax_opt.plot([], [], marker='.', color=colors_opt[name], alpha=0.8, lw=2, label=f"{name} (LR={opt_lrs[name]})")[0] for name in paths_opt.keys()]
    setup_plot('Comparing Advanced Optimizers Simultaneously (2D Non-Convex)', 'x', 'y', ax_opt)
    ax_opt.legend()
    def animate_opt(frame):
        for idx, name in enumerate(paths_opt.keys()):
            px, py = paths_opt[name][0][:frame+1], paths_opt[name][1][:frame+1]
            lines_opt[idx].set_data(px, py)
        return lines_opt
    anim_opt = FuncAnimation(fig_opt, animate_opt, frames=epochs_opt+1, interval=150, blit=True)
    plt.close()
    return anim_opt

def plot_l1_l2_geometric(lambda_val, center_w1, center_w2):
    w1 = np.linspace(-2, 3, 100)
    w2 = np.linspace(-2, 3, 100)
    W1, W2 = np.meshgrid(w1, w2)
    loss_contour = (W1 - center_w1)**2 + 2*(W2 - center_w2)**2
    radius = max(0.1, 2.5 - lambda_val * 2.0)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].contour(W1, W2, loss_contour, levels=[0.2, 0.8, 1.5, 2.5, 4.0, 6.0], colors='gray', alpha=0.7)
    axes[0].plot([0, radius, 0, -radius, 0], [radius, 0, -radius, 0, radius], color='red', lw=2.5, label=f'L1 Constraint (r={radius:.2f})')
    intersection_l1_x, intersection_l1_y = 0.0, radius
    if abs(center_w1) > abs(center_w2):
        intersection_l1_x, intersection_l1_y = radius * np.sign(center_w1), 0.0
    axes[0].scatter([intersection_l1_x], [intersection_l1_y], color='blue', s=120, zorder=5)
    setup_plot('L1 Regularization (Diamond Constraint)', 'w1', 'w2', axes[0])
    axes[0].axhline(0, color='black', lw=0.8); axes[0].axvline(0, color='black', lw=0.8)
    axes[0].scatter([center_w1], [center_w2], color='black', s=100, marker='*', label='Unregularized Opt'); axes[0].legend()
    axes[1].contour(W1, W2, loss_contour, levels=[0.2, 0.8, 1.5, 2.5, 4.0, 6.0], colors='gray', alpha=0.7)
    circle = plt.Circle((0, 0), radius, color='red', fill=False, lw=2.5, label=f'L2 Constraint (r={radius:.2f})')
    axes[1].add_artist(circle)
    mag = np.sqrt(center_w1**2 + center_w2**2)
    intersection_l2_x = radius * (center_w1 / mag) if mag > 0 else 0.0
    intersection_l2_y = radius * (center_w2 / mag) if mag > 0 else 0.0
    axes[1].scatter([intersection_l2_x], [intersection_l2_y], color='blue', s=120, zorder=5)
    setup_plot('L2 Regularization (Circular Constraint)', 'w1', 'w2', axes[1])
    axes[1].axhline(0, color='black', lw=0.8); axes[1].axvline(0, color='black', lw=0.8)
    axes[1].scatter([center_w1], [center_w2], color='black', s=100, marker='*', label='Unregularized Opt'); axes[1].legend()
    plt.tight_layout(); plt.show()

def plot_curve_augmentation(num_aug, noise_level):
    np.random.seed(42)
    x_true = np.linspace(-3, 3, 200)
    y_true = np.sin(x_true)
    x_train = np.random.uniform(-3, 3, 15)
    y_train = np.sin(x_train) + np.random.normal(0, 0.25, 15)
    x_aug_list, y_aug_list = [], []
    if num_aug > 0:
        indices = np.random.choice(len(x_train), num_aug, replace=True)
        x_aug_list = x_train[indices] + np.random.normal(0, noise_level, num_aug)
        y_aug_list = y_train[indices] + np.random.normal(0, 0.15, num_aug)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x_true, y_true, 'k--', label='True function (sin(x))', alpha=0.5)
    ax.scatter(x_train, y_train, color='red', s=80, label='Original Data (N=15)', zorder=5)
    if num_aug > 0:
        ax.scatter(x_aug_list, y_aug_list, color='blue', marker='x', s=45, alpha=0.6, label=f'Augmented Data (N={num_aug})')
    p_noreg = np.polyfit(x_train, y_train, 9)
    y_fit_noreg = np.polyval(p_noreg, x_true)
    ax.plot(x_true, y_fit_noreg, color='red', lw=2, label='Fit WITHOUT Augmentation (Overfitted)')
    if num_aug > 0:
        x_combined = np.concatenate([x_train, x_aug_list])
        y_combined = np.concatenate([y_train, y_aug_list])
        p_aug = np.polyfit(x_combined, y_combined, 9)
        y_fit_aug = np.polyval(p_aug, x_true)
        ax.plot(x_true, y_fit_aug, color='blue', lw=2.5, label='Fit WITH Augmentation (Regularized)')
    setup_plot('Regularization via Noise-based Data Augmentation', 'x', 'y', ax)
    ax.set_ylim(-2.0, 2.0); ax.legend(loc='lower right'); plt.show()

def plot_mnist_samples(subset):
    fig, axes = plt.subplots(2, 5, figsize=(10, 5))
    for i, ax in enumerate(axes.flat):
        img, label = subset[i]
        ax.imshow(img.squeeze().numpy(), cmap='gray')
        ax.set_title(f"Label: {label}")
        ax.axis('off')
    fig.suptitle("Sample Images from MNIST Dataset (28x28)", fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

def plot_neural_network(ax, layer_sizes):
    ax.clear()
    ax.axis('off')
    v_spacing = 0.8
    h_spacing = 1.5
    layer_sizes_org=layer_sizes
    layer_sizes=[16,8,4,2]
    max_nodes = 8

    for n, layer_size in enumerate(layer_sizes):
        layer_name = f"Layer {n}\n({layer_sizes_org[n]} units)"
        if n == 0:
            layer_name = f"Input\n({layer_sizes_org[n]} units)"
        elif n == len(layer_sizes)-1:
            layer_name = f"Output\n({layer_sizes_org[n]} units)"

        ax.text(n * h_spacing, -1.0, layer_name, ha='center', va='bottom', fontsize=10, fontweight='bold')

        nodes_to_draw = min(layer_size, max_nodes)
        y_offset = (max_nodes - nodes_to_draw) / 2.0

        for m in range(nodes_to_draw):
            x = n * h_spacing
            y = (m + y_offset) * v_spacing
            circle = plt.Circle((x, y), 0.1, color='skyblue', ec='black', zorder=4)
            ax.add_artist(circle)

            # if layer_size > max_nodes and m == nodes_to_draw // 2:
                # ax.text(x, y, "...", ha='center', va='center', fontsize=12, fontweight='bold', color='black')
            # if layer_size < layer_sizes_org[n] and m == nodes_to_draw // 2:
                # ax.text(x, y-y_offset* v_spacing, "...", ha='center', va='center', fontsize=12, fontweight='bold', color='black')

            if n < len(layer_sizes) - 1:
                next_layer_size = layer_sizes[n+1]
                next_nodes_to_draw = min(next_layer_size, max_nodes)
                next_y_offset = (max_nodes - next_nodes_to_draw) / 2.0
                for o in range(next_nodes_to_draw):
                    nx = (n + 1) * h_spacing
                    ny = (o + next_y_offset) * v_spacing
                    ax.plot([x, nx], [y, ny], color='gray', alpha=0.3, zorder=1)

    ax.set_xlim(-0.5, (len(layer_sizes) - 0.5) * h_spacing)
    ax.set_ylim(-0.5, max_nodes * v_spacing)


def plot_lr_schedulers_comparison(sched_losses, sched_lrs, names):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for name in names:
        axes[0].plot(sched_losses[name], label=name, lw=2)
    setup_plot('Training Loss with Different Schedulers', 'Epoch', 'Loss', axes[0]); axes[0].legend()
    for name in names:
        axes[1].plot(sched_lrs[name], label=name, lw=2)
    setup_plot('Learning Rate Decay Schedule', 'Epoch', 'Learning Rate', axes[1]); axes[1].legend()
    plt.tight_layout(); plt.show()

def plot_dropout_validation_loss(do_val_losses, rates, colors):
    fig, ax = plt.subplots(figsize=(10, 5))
    for p, c in zip(rates, colors):
        ax.plot(do_val_losses[p], label=f'Dropout p={p}', color=c, lw=2)
    setup_plot('Impact of Dropout on Validation Loss', 'Epoch', 'Validation Loss', ax); ax.legend(); plt.show()


def plot_weight_histograms(ax, w_noreg, w_l1, w_l2):
    ax.clear()
    ax.hist(w_noreg, bins=50, alpha=0.4, label='No Regularization', color='blue', edgecolor='blue')
    ax.hist(w_l1, bins=50, alpha=0.4, label='L1 Regularization', color='green', edgecolor='green')
    ax.hist(w_l2, bins=50, alpha=0.4, label='L2 Regularization (Weight Decay)', color='orange', edgecolor='orange')
    ax.set_title('Impact of L1 vs L2 Regularization on Weights', fontsize=11, fontweight='bold')
    ax.set_xlabel('Weight Value')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

def plot_dropout_activations(axes, p_rates, sample_input):
    for i, p in enumerate(p_rates):
        # Create temporary linear layer + dropout to inspect activations
        layer = nn.Linear(784, 128)
        # Use fixed seed for weight matrix so weights are identical across dropout tests
        torch.manual_seed(42)
        nn.init.xavier_normal_(layer.weight)
        nn.init.zeros_(layer.bias)

        dropout = nn.Dropout(p)

        with torch.no_grad():
            act = torch.relu(layer(sample_input))
            act_dropped = dropout(act).numpy().flatten()

        grid = act_dropped.reshape(8, 16)
        ax = axes[i]
        ax.clear()
        im = ax.imshow(grid, cmap='viridis', aspect='auto', interpolation='nearest')
        ax.set_title(f"Dropout p={p}\n(Active: {(act_dropped > 0).sum()}/128)", fontsize=10, fontweight='bold')
        ax.axis('off')

def plot_augmentation_generalization_gap(train_noreg, val_noreg, train_aug, val_aug):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_noreg, label='Without Augmentation - Train', color='red', linestyle='--', lw=2)
    ax.plot(val_noreg, label='Without Augmentation - Val', color='red', linestyle='-', lw=2)
    ax.plot(train_aug, label='With Augmentation - Train', color='blue', linestyle='--', lw=2)
    ax.plot(val_aug, label='With Augmentation - Val', color='blue', linestyle='-', lw=2)
    setup_plot('Data Augmentation: Generalization Gap Comparison', 'Epoch', 'Cross Entropy Loss', ax); ax.legend(); plt.show()

def plot_bias_variance(epochs, train_l, val_l, d_ll, d_hb, d_lh, d_hh):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].plot(epochs, train_l, label='Training Loss', color='blue', lw=2)
    axes[0].plot(epochs, val_l, label='Validation Loss', color='red', lw=2)
    axes[0].axvline(40, color='green', linestyle='--', label='Optimal Capacity')
    axes[0].text(10, 0.6, 'High Bias\n(Underfitting)', ha='center', fontsize=10, fontweight='bold')
    axes[0].text(80, 0.6, 'High Variance\n(Overfitting)', ha='center', fontsize=10, fontweight='bold')
    setup_plot('Loss Curves: Bias-Variance Tradeoff', 'Model Complexity / Epochs', 'Error', axes[0]); axes[0].legend()
    from matplotlib.patches import Circle
    def draw_bullseye(ax, center_x, center_y, title, darts):
        for r, c in zip([0.4, 0.3, 0.2, 0.1], ['#ff9999', '#ff6666', '#ff3333', '#cc0000']):
            ax.add_patch(Circle((center_x, center_y), r, color=c, ec='black'))
        ax.scatter(darts[0], darts[1], color='blue', s=40, zorder=5)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlim(center_x-0.5, center_x+0.5); ax.set_ylim(center_y-0.5, center_y+0.5); ax.axis('off')
    axes[1].axis('off')
    ax_ll = fig.add_axes([0.55, 0.55, 0.2, 0.35])
    ax_hb = fig.add_axes([0.75, 0.55, 0.2, 0.35])
    ax_lh = fig.add_axes([0.55, 0.15, 0.2, 0.35])
    ax_hh = fig.add_axes([0.75, 0.15, 0.2, 0.35])
    draw_bullseye(ax_ll, 0.5, 0.5, "Low Bias, Low Var", d_ll)
    draw_bullseye(ax_hb, 0.5, 0.5, "High Bias, Low Var", d_hb)
    draw_bullseye(ax_lh, 0.5, 0.5, "Low Bias, High Var", d_lh)
    draw_bullseye(ax_hh, 0.5, 0.5, "High Bias, High Var", d_hh)
    fig.suptitle('2D Concept of Bias and Variance', x=0.75, y=0.98, fontsize=14, fontweight='bold')
    plt.show()


# Part 1: Interactive Playgrounds (Toy Data & Concepts)

In this section, we explore core deep learning optimization and regularization concepts using simple, visualizable 1D/2D toy functions and synthetic landscapes.


## 1. Convex vs Non-Convex Functions

In optimization, the landscape of our loss function dictates how easily we can find the optimal parameters.
- **Convex Functions:** Have a single global minimum. Any local minimum is the global minimum.
- **Non-Convex Functions:** Can have multiple local minima, making it challenging for gradient descent to find the absolute best (global) solution.


In [7]:
interact(
    plot_1d_contour_surface_viewer,
    expr_1d=widgets.Text(value="x**2", description="1D Expr:", continuous_update=False),
    expr_2d=widgets.Text(value="x**2+y**2", description="2D Expr:", continuous_update=False)
)


interactive(children=(Text(value='x**2', continuous_update=False, description='1D Expr:'), Text(value='x**2+y*…

<function __main__.plot_1d_contour_surface_viewer(expr_1d, expr_2d)>

In [6]:
interact(
    plot_1d_contour_surface_viewer,
    expr_1d=widgets.Text(value="x**2 + 10", description="1D Expr:", continuous_update=False),
    expr_2d=widgets.Text(value="x**2 + y**2 + 10", description="2D Expr:", continuous_update=False)
)


interactive(children=(Text(value='x**2 + 10', continuous_update=False, description='1D Expr:'), Text(value='x*…

<function __main__.plot_1d_contour_surface_viewer(expr_1d, expr_2d)>